In [ ]:
# Cài đặt datasets phiên bản cũ (2.19.0) để hỗ trợ load script conll2003
!pip install datasets==2.19.0 seqeval -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import os

# 1. Thiết lập thiết bị
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

# 2. Tải dữ liệu thủ công (Nguồn GitHub mirror của CoNLL 2003)
# eng.train: Tập huấn luyện
# eng.testa: Tập validation (Dev)
# eng.testb: Tập test
!wget https://raw.githubusercontent.com/glample/tagger/master/dataset/eng.train -O train.txt
!wget https://raw.githubusercontent.com/glample/tagger/master/dataset/eng.testa -O dev.txt
!wget https://raw.githubusercontent.com/glample/tagger/master/dataset/eng.testb -O test.txt

print("Đã tải xong dữ liệu train.txt, dev.txt, test.txt")

# 3. Hàm đọc file CoNLL thủ công
def load_conll_manual(file_path):
    sentences = []
    current_tokens = []
    current_tags = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # Nếu dòng trống -> Hết 1 câu
            if not line:
                if current_tokens:
                    sentences.append({'tokens': current_tokens, 'ner_tags': current_tags})
                    current_tokens = []
                    current_tags = []
                continue

            # Dòng dữ liệu: "EU NNP I-NP I-ORG"
            # Cấu trúc: Word POS Chunk NER
            parts = line.split()
            if len(parts) >= 4: # Đảm bảo đủ cột
                word = parts[0]
                ner_tag = parts[3] # Cột cuối cùng là NER label

                current_tokens.append(word)
                current_tags.append(ner_tag)

    # Thêm câu cuối nếu còn sót
    if current_tokens:
        sentences.append({'tokens': current_tokens, 'ner_tags': current_tags})

    return sentences

# Load dữ liệu vào biến
train_data = load_conll_manual('train.txt')
val_data = load_conll_manual('dev.txt')

print(f"Số câu Train: {len(train_data)}")
print(f"Số câu Val: {len(val_data)}")
print("Mẫu dữ liệu:", train_data[0])

Đang sử dụng thiết bị: cpu
--2025-12-06 03:29:01--  https://raw.githubusercontent.com/glample/tagger/master/dataset/eng.train
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3281527 (3.1M) [text/plain]
Saving to: ‘train.txt’

train.txt           100%[===================>]   3.13M  --.-KB/s    in 0.08s   

2025-12-06 03:29:01 (37.8 MB/s) - ‘train.txt’ saved [3281527/3281527]

--2025-12-06 03:29:01--  https://raw.githubusercontent.com/glample/tagger/master/dataset/eng.testa
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 827009 (808

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

def build_vocab_and_tags_manual(data):
    print("Đang xây dựng từ điển...")
    word_counter = Counter()
    tag_set = set()

    for item in data:
        for token in item['tokens']:
            word_counter[token] += 1
        for tag in item['ner_tags']:
            tag_set.add(tag)

    # 1. Word to Index
    word_to_ix = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for word, _ in word_counter.most_common():
        word_to_ix[word] = len(word_to_ix)

    # 2. Tag to Index (Map Tag chữ sang số)
    # Đảm bảo PAD ở vị trí 0
    tag_to_ix = {PAD_TOKEN: PAD_IDX}
    for tag in sorted(list(tag_set)):
        tag_to_ix[tag] = len(tag_to_ix)

    return word_to_ix, tag_to_ix

# Xây dựng từ điển từ tập Train
word_to_ix, tag_to_ix = build_vocab_and_tags_manual(train_data)

print(f"Vocab size: {len(word_to_ix)}")
print(f"Tag size: {len(tag_to_ix)}")
print(f"Danh sách Tags: {tag_to_ix}")

Đang xây dựng từ điển...
Vocab size: 23626
Tag size: 9
Danh sách Tags: {'<PAD>': 0, 'B-LOC': 1, 'B-MISC': 2, 'B-ORG': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


In [ ]:
class NERDatasetManual(Dataset):
    def __init__(self, data, word_to_ix, tag_to_ix):
        self.data = data
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item['tokens']
        tags_str = item['ner_tags'] # Đang là chữ ['I-ORG', 'O']

        # Word -> Index
        word_indices = [self.word_to_ix.get(w, UNK_IDX) for w in tokens]

        # Tag -> Index
        tag_indices = [self.tag_to_ix[t] for t in tags_str]

        return torch.tensor(word_indices), torch.tensor(tag_indices)

def collate_fn(batch):
    sentences, tags = zip(*batch)
    padded_sentences = pad_sequence(sentences, batch_first=True, padding_value=PAD_IDX)
    padded_tags = pad_sequence(tags, batch_first=True, padding_value=PAD_IDX)
    return padded_sentences, padded_tags

# Tạo DataLoader
BATCH_SIZE = 32
train_dataset = NERDatasetManual(train_data, word_to_ix, tag_to_ix)
val_dataset = NERDatasetManual(val_data, word_to_ix, tag_to_ix)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
class BiLSTMNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pad_idx, dropout_rate=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # Bi-LSTM
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout_rate)

        # Linear (hidden * 2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, _ = self.lstm(embedded)
        outputs = self.dropout(outputs)
        predictions = self.fc(outputs)
        return predictions

In [ ]:
# Cấu hình
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = len(tag_to_ix)
INPUT_DIM = len(word_to_ix)
N_EPOCHS = 5
LEARNING_RATE = 0.001

model = BiLSTMNER(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, PAD_IDX)
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Hàm Train & Eval (Ngắn gọn)
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0
    for batch in iterator:
        text, tags = batch
        text, tags = text.to(device), tags.to(device)
        optimizer.zero_grad()
        predictions = model(text)
        loss = criterion(predictions.view(-1, OUTPUT_DIM), tags.view(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    correct_tokens, total_tokens = 0, 0
    with torch.no_grad():
        for batch in iterator:
            text, tags = batch
            text, tags = text.to(device), tags.to(device)
            predictions = model(text)
            loss = criterion(predictions.view(-1, OUTPUT_DIM), tags.view(-1))
            epoch_loss += loss.item()

            preds = torch.argmax(predictions, dim=-1)
            mask = (tags != PAD_IDX)
            correct_tokens += ((preds == tags) & mask).sum().item()
            total_tokens += mask.sum().item()
    return epoch_loss / len(iterator), correct_tokens / total_tokens

# --- MAIN LOOP ---
print("Bắt đầu huấn luyện Bi-LSTM NER (Manual Load)...")
for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, val_loader, criterion)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val Loss: {valid_loss:.3f} | Val Acc: {valid_acc*100:.2f}%')

# Lưu model
torch.save(model.state_dict(), 'best_ner_model_manual.pt')

Bắt đầu huấn luyện Bi-LSTM NER (Manual Load)...
Epoch: 01 | Train Loss: 0.563 | Val Loss: 0.412 | Val Acc: 87.41%
Epoch: 02 | Train Loss: 0.383 | Val Loss: 0.307 | Val Acc: 90.09%
Epoch: 03 | Train Loss: 0.306 | Val Loss: 0.241 | Val Acc: 92.27%
Epoch: 04 | Train Loss: 0.258 | Val Loss: 0.204 | Val Acc: 93.50%
Epoch: 05 | Train Loss: 0.223 | Val Loss: 0.188 | Val Acc: 93.81%


In [ ]:
def predict_ner(sentence):
    model.load_state_dict(torch.load('best_ner_model_manual.pt'))
    model.eval()
    tokens = sentence.split()
    indices = [word_to_ix.get(t, UNK_IDX) for t in tokens]
    tensor = torch.LongTensor(indices).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model(tensor)
        max_preds = prediction.argmax(dim=-1).squeeze(0)

    ix_to_tag = {v: k for k, v in tag_to_ix.items()}
    tags = [ix_to_tag[ix.item()] for ix in max_preds]

    print(f"\nCâu: {sentence}")
    print(list(zip(tokens, tags)))

predict_ner("VNU University is located in Hanoi")


Câu: VNU University is located in Hanoi
[('VNU', 'I-ORG'), ('University', 'I-ORG'), ('is', 'O'), ('located', 'O'), ('in', 'O'), ('Hanoi', 'I-LOC')]
